In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import src.visualization
importlib.reload(src.visualization)
from src.visualization import plot_pca, plot_umap, plot_tsne, plot_pca_dark, plot_umap_dark, plot_tsne_dark

# # ---- palette (one place, change here) ----
# PALETTE = "pastel"            # try "muted", "pastel", "Set2", "tab10", "deep"
# sns.set_palette(PALETTE)


#### benchmarked90 (260603 run): `proof-benchmarked-p90.yaml`

Single-dataset deep-dive (feature matrix, PCA/UMAP, greedy-FS comparison) on the p90 amortized SPI set. Re-pointed from the retired `proof_benchmarked_260505` run.

In [ ]:
data_p90 = np.load("../../features/data-embeddings-proof_benchmarked90_260603_pearson.npz", allow_pickle=True)
X = data_p90["X"]
y = data_p90["y"]
M = data_p90["M"].astype(int)
T = data_p90["T"].astype(int)
instance = data_p90["instance"]
dataset_paths = data_p90["dataset_paths"]

# --- filters (set to None to disable) ---
M_filter = None
classes = [
            "cauchy-noise", "gaussian-noise", 
            "kuramoto_omega-fast","kuramoto_omega-slow", 
            "defect-turbulence", "sti-i",
            "var-phi-0.2_cpl-0.4",
            # "var-phi-0.2_cpl-0.8",
            "var-phi-0.95_cpl-0.4",
            "wave-1d"
            ]

mask = np.ones(len(y), dtype=bool)
if M_filter is not None:
    mask &= M == M_filter
if classes is not None:
    mask &= np.isin(y, classes)

X = X[mask]
y = y[mask]
M = M[mask]
T = T[mask]
instance = instance[mask]
dataset_paths = dataset_paths[mask]

meta_df = pd.DataFrame({
    "mts_class": y,
    "M": M,
    "T": T,
    "instance": instance,
    "dataset_paths": dataset_paths,
})
print(f"X: {X.shape}, classes: {np.unique(y).tolist()}")

##### Feature matrix

In [ ]:
import seaborn as sns

# sort rows by class so blocks are visible
order = np.argsort(y)
X_sorted = X[order]
y_sorted = y[order]

fig, ax = plt.subplots(figsize=(16, 6), dpi=150)
ax.imshow(X_sorted, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1, interpolation="none")
ax.set_xlabel("SPI-SPI feature index")
ax.set_ylabel("dataset")
ax.set_yticks([])
ax.set_title(f"Feature matrix ({X_sorted.shape[0]} x {X_sorted.shape[1]})")

# class boundaries
boundaries = np.where(np.diff(np.searchsorted(np.unique(y_sorted), y_sorted)))[0] + 0.5
for b in boundaries:
    ax.axhline(b, color="white", linewidth=0.5)

plt.tight_layout()
plt.show()

##### PCA - 2D

In [ ]:
alpha=0.5
s=40
kde=False
ax=True

fig, ax = plt.subplots(figsize=(10, 7), dpi=500)
plot_pca(X, meta_df, alpha=alpha, s=s, ax=ax, kde=kde)
plt.tight_layout(); plt.show()

##### PCA - 3D

In [ ]:
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 3D PCA on scaled features (matches plot_pca's StandardScaler step).
X_scaled = StandardScaler().fit_transform(X)
pca3 = PCA(n_components=3, random_state=0)
emb3 = pca3.fit_transform(X_scaled)
var3 = pca3.explained_variance_ratio_

df3 = meta_df.assign(
    pc1=emb3[:, 0], pc2=emb3[:, 1], pc3=emb3[:, 2],
    instance_str=meta_df["instance"].astype(str),
    dataset_paths_str=meta_df["dataset_paths"].astype(str),
)

fig = px.scatter_3d(
    df3, x="pc1", y="pc2", z="pc3",
    color="mts_class",
    size="M", size_max=16,
    hover_data={
        "mts_class": True, "M": True, "T": True,
        "instance_str": True, "dataset_paths_str": False,
        "pc1": ":.2f", "pc2": ":.2f", "pc3": ":.2f",
    },
    title=f"PCA 3D | Var: {var3.sum():.4f}  (PC1 {var3[0]:.3f}, PC2 {var3[1]:.3f}, PC3 {var3[2]:.3f})",
    width=1200, height=900,
)
fig.update_traces(marker=dict(opacity=0.55))
fig.update_layout(legend=dict(title="mts_class"),
                  scene=dict(xaxis_title=f"PC1 ({var3[0]:.3f})",
                             yaxis_title=f"PC2 ({var3[1]:.3f})",
                             zaxis_title=f"PC3 ({var3[2]:.3f})",
                             aspectmode="cube"))
fig.show()


##### UMAP - 2D

In [ ]:
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from umap import UMAP

# Shared UMAP hyperparameters.
n_neighbors = 12
min_dist = 0.75
metric = "correlation"   # "euclidean" | "correlation" | "cosine"
scaled = False           # True -> StandardScaler(X); False -> raw X

X_input = StandardScaler().fit_transform(X) if scaled else X
emb = UMAP(n_neighbors=n_neighbors, min_dist=min_dist, metric=metric,
                  random_state=0, verbose=False).fit_transform(X_input)

df_emb = meta_df.assign(umap_x=emb[:, 0], umap_y=emb[:, 1])

fig, ax = plt.subplots(figsize=(8, 6), dpi=500)
sns.kdeplot(data=df_emb, x="umap_x", y="umap_y", hue="mts_class",
            levels=5, alpha=0.3, linewidths=1, ax=ax, legend=False)
sns.scatterplot(data=df_emb, x="umap_x", y="umap_y",
                hue="mts_class", size="M",
                alpha=0.3, edgecolor="face", ax=ax, legend="full")
ax.set_title(f"UMAP (NN={n_neighbors}, min_dist={min_dist}, metric={metric}, scaled={scaled})")
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.set_box_aspect(1)
ax.legend(title="mts_class", loc="upper left", bbox_to_anchor=(1.02, 1),
          fontsize=9, frameon=True)
plt.tight_layout()
plt.savefig("../../notebooks/embeddings/graphics/umap_benchmarked90_260603.svg", dpi=500, bbox_inches="tight")
plt.show()

In [ ]:
import plotly.express as px

# Interactive version of the chosen combo (emb_chosen / chosen_label from the cell above).
df_plot = meta_df.assign(
    umap_x=emb[:, 0],
    umap_y=emb[:, 1],
    M_str=meta_df["M"].astype(str),
    instance_str=meta_df["instance"].astype(str),
    dataset_paths_str=meta_df["dataset_paths"].astype(str),
)
fig = px.scatter(
    df_plot, x="umap_x", y="umap_y",
    color="mts_class",
    symbol="M_str",
    hover_data={
        "mts_class": True,
        "M": True,
        "T": True,
        "instance_str": True,
        "dataset_paths_str": False,
        "M_str": False,
        "umap_x": ":.2f",
        "umap_y": ":.2f",
    },
    title=f"UMAP (NN={n_neighbors}, min_dist={min_dist}, metric={metric}, {'scaled' if scaled else 'raw'})",
    width=900, height=700,
)
fig.update_traces(marker=dict(size=9, opacity=0.75))
fig.update_layout(legend=dict(title="mts_class / M"))
fig.show()

##### UMAP - 3D

##### **Comparison with reduced feature set from Greedy FS**

In [ ]:
import seaborn as sns



In [ ]:
data_greedy = np.load("../../features/data-embeddings-proof_benchmarked90_260603_pearson_final_S.txt.npz", allow_pickle=True)
greedy_idx = pd.Series(
    np.arange(len(data_greedy["dataset_paths"])),
    index=data_greedy["dataset_paths"]
).loc[dataset_paths].to_numpy()

X_greedy = data_greedy["X"][greedy_idx]
assert np.array_equal(data_greedy["dataset_paths"][greedy_idx], dataset_paths)
compare_X = {"full-benchmarked90": X, "greedy-fs": X_greedy}
print(f"Full: {X.shape} | greedy-fs: {X_greedy.shape}")

# sort rows by class so blocks are visible
order = np.argsort(y)
Xg_sorted = X_greedy[order]
y_sorted = y[order]

fig, ax = plt.subplots(figsize=(8, 3), dpi=500)

im = ax.imshow(
    Xg_sorted,
    aspect="auto",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    interpolation="none",
)

ax.set_xlabel("greedy SPI-SPI feature index")
ax.set_ylabel("dataset")
ax.set_yticks([])
ax.set_title(f"Greedy feature matrix ({Xg_sorted.shape[0]} x {Xg_sorted.shape[1]})")

# class boundaries
class_codes = np.searchsorted(np.unique(y_sorted), y_sorted)
boundaries = np.where(np.diff(class_codes))[0] + 0.5

for b in boundaries:
    ax.axhline(b, color="white", linewidth=0.5)

fig.colorbar(im, ax=ax, label="feature value")
plt.tight_layout()
plt.show()


alpha = 0.5
s = 40
kde = False

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=500)
for ax, (label, Xi) in zip(axes, compare_X.items()):
    plot_pca(Xi, meta_df, alpha=alpha, s=s, ax=ax, kde=kde, feature_space=label)
    ax.set_title(ax.get_title().replace(f"({label})", label))
axes[0].get_legend().remove()
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from umap import UMAP

n_neighbors = 12
min_dist = 0.75
metric = "correlation"
scaled = False
umap_compare = {}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=500)
for ax, (label, Xi) in zip(axes, compare_X.items()):
    X_input = StandardScaler().fit_transform(Xi) if scaled else Xi
    emb_i = UMAP(n_neighbors=n_neighbors, min_dist=min_dist, metric=metric,
                 random_state=0, verbose=False).fit_transform(X_input)
    umap_compare[label] = emb_i
    df_emb = meta_df.assign(umap_x=emb_i[:, 0], umap_y=emb_i[:, 1])
    sns.kdeplot(data=df_emb, x="umap_x", y="umap_y", hue="mts_class",
                levels=5, alpha=0.3, linewidths=1, ax=ax, legend=False)
    sns.scatterplot(data=df_emb, x="umap_x", y="umap_y",
                    hue="mts_class", size="M",
                    alpha=0.3, edgecolor="face", ax=ax, legend="full")
    ax.set_title(f"{label} | UMAP")
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    ax.set_box_aspect(1)
axes[0].get_legend().remove()
axes[1].legend(title="mts_class", loc="upper left", bbox_to_anchor=(1.02, 1),
               fontsize=9, frameon=True)
plt.tight_layout(); plt.show()


In [ ]:
from plotly.subplots import make_subplots
import plotly.express as px

fig = make_subplots(rows=1, cols=2, subplot_titles=list(umap_compare))
for col, (label, emb_i) in enumerate(umap_compare.items(), start=1):
    df_plot = meta_df.assign(
        umap_x=emb_i[:, 0],
        umap_y=emb_i[:, 1],
        M_str=meta_df["M"].astype(str),
        instance_str=meta_df["instance"].astype(str),
        dataset_paths_str=meta_df["dataset_paths"].astype(str),
    )
    panel = px.scatter(
        df_plot, x="umap_x", y="umap_y",
        color="mts_class", symbol="M_str",
        hover_data={
            "mts_class": True,
            "M": True,
            "T": True,
            "instance_str": True,
            "dataset_paths_str": False,
            "M_str": False,
            "umap_x": ":.2f",
            "umap_y": ":.2f",
        },
    )
    for trace in panel.data:
        trace.showlegend = col == 1
        fig.add_trace(trace, row=1, col=col)
fig.update_traces(marker=dict(size=9, opacity=0.75))
fig.update_xaxes(title_text="UMAP-1")
fig.update_yaxes(title_text="UMAP-2")
fig.update_layout(
    title=f"UMAP comparison (NN={n_neighbors}, min_dist={min_dist}, metric={metric}, {'scaled' if scaled else 'raw'})",
    width=1100, height=550,
    legend=dict(title="mts_class / M"),
)
fig.show()
